In [33]:
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from datasets import load_from_disk

In [15]:
checkpoint_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_id)

In [25]:
tokenized_datasets = load_from_disk("./dataset/tokenized_datasets")

In [26]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21385
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5347
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3432
    })
})

In [28]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [29]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
  f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]

  return {"accuracy": acc, "f1": f1}


In [30]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_id, num_labels=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
training_args = TrainingArguments(
    output_dir="bert-twitter-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.441368,0.513797,0.793155,0.793464
2,0.316953,0.593155,0.785861,0.785082


Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1338, training_loss=0.38260822481460455, metrics={'train_runtime': 302.6665, 'train_samples_per_second': 141.311, 'train_steps_per_second': 4.421, 'total_flos': 2957065923490560.0, 'train_loss': 0.38260822481460455, 'epoch': 2.0})

In [35]:
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}

model.config.id2label = id2label
model.config.label2id = label2id
local_model_path = "./bert-twitter-sentiment"


In [36]:
trainer.save_model(local_model_path)
classifier = pipeline(
    "text-classification",
    model=local_model_path,
    device_map="auto"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [37]:
test_tweets = [
    "I absolutely love the new features on this app, amazing job!",
    "This update completely ruined my day. Worst interface ever."
]

results = classifier(test_tweets)

for tweet, result in zip(test_tweets, results):
    print(f"Tweet: '{tweet}'\nPrediction: {result['label']} | Confidence: {result['score']:.2%}\n")

Tweet: 'I absolutely love the new features on this app, amazing job!'
Prediction: Positive | Confidence: 98.59%

Tweet: 'This update completely ruined my day. Worst interface ever.'
Prediction: Negative | Confidence: 99.06%

